In [34]:
from OSMPythonTools.overpass import Overpass, overpassQueryBuilder
import pandas as pd
from tqdm.auto import tqdm

/home/miro/Projects/School/VolcanoReadiness/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
overpass = Overpass()

In [3]:
def get_volcanoes(lat_band=10):
    query = overpassQueryBuilder(
        bbox=[-lat_band, -180, lat_band, 180],   # [south, west, north, east]
        elementType=['node', 'way', 'relation'],
        selector=['"natural"="volcano"'],
        includeCenter=True,   # gives ways/relations a representative point
        out='body',
    )
    result = overpass.query(query, timeout=180)

    rows = []
    for el in result.elements():
        if el.type() == 'node':
            lat, lon = el.lat(), el.lon()
        else:
            lat, lon = el.centerLat(), el.centerLon()
        rows.append({
            'osm_type': el.type(),
            'osm_id': el.id(),
            'name': el.tag('name'),
            'name_en': el.tag('name:en'),
            'lat': lat,
            'lon': lon,
            'elevation': el.tag('ele'),
            'volcano_type': el.tag('volcano:type'),
            'volcano_status': el.tag('volcano:status'),
            'wikidata': el.tag('wikidata'),
        })
    return pd.DataFrame(rows)

In [7]:
df = get_volcanoes()

In [23]:
df = df[df["name"].apply(lambda x: isinstance(x,str))]

### Ok, it looks like there's a lot of volcanoes where the volcano status tag is missing...
Maybe we need a separate DB to filter... Whatever, for now, let's roll with this

In [27]:
df[df['volcano_status'] == 'active'].sample(5)

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata
1147,node,292801401,Gunung Batur,Mount Batur,-8.239415,115.377960,1717,caldera,active,Q43876
373,node,319478134,Mount Makaturing,NaN,7.654532,124.303132,1940,stratovolcano,active,Q1586051
514,node,332021231,Mount Piapayungan,NaN,7.690834,124.507222,2802,stratovolcano,active,Q31478904
522,node,361305801,Azufral,NaN,1.085538,-77.718588,4070,stratovolcano,active,Q794264
595,node,545786074,Gunung Gamkonora,Mount Gamkonora,1.379072,127.532281,1571,NaN,active,Q1493157


In [38]:
from find_volcano import find_volcano_smithsonian, load_gvmid_stations

ImportError: cannot import name 'load_gvmid_stations' from 'find_volcano' (/home/miro/Projects/School/VolcanoReadiness/find_volcano.py)

In [29]:
find_volcano_smithsonian("Acatenango")

,Volcano_Number,Volcano_Name,Volcanic_Landform,Last_Eruption_Year,Country,Latitude,Longitude
0,342080,Acatenango,Composite,1972,Guatemala,14.501,-90.876


In [36]:
extra_df = pd.concat(
    [
        find_volcano_smithsonian(name).rename(columns={"Volcano_Name": "name", "Volcano_Number": "volcano_number", "Country": "country", "Volcanic_Landform":"volcanic_landform", "Last_Eruption_Year":"last_eruption_year"})
        for name in tqdm(df["name"].unique(), desc="Fetching info")
    ],
    ignore_index=True
)

extra_df = extra_df[["name", "volcano_number", "country", "volcanic_landform", "last_eruption_year"]]

df_merged = df.merge(extra_df, on="Name", how="left")

Fetching info: 100%|██████████████████████████████| 593/593 [06:22<00:00,  1.55it/s]


KeyError: 'Name'

In [39]:
extra_df

,name,volcano_number,country,volcanic_landform,last_eruption_year
0,Chimborazo,352071,Ecuador,Composite,550
1,Atacazo,352021,Ecuador,Composite,-320
2,Imbabura,352004,Ecuador,Composite,-5550
3,Chiles-Cerro Negro,351110,Colombia-Ecuador,Composite,1936
4,Reventador,352010,Ecuador,Composite,2026
...,...,...,...,...,...
79,Toliman,342070,Guatemala,Composite,None
80,Takuan Group,255021,Papua New Guinea,Composite,None
81,Billy Mitchell,255011,Papua New Guinea,Caldera,1580
82,Bilate River Field,221291,Ethiopia,Cluster,None


In [41]:
df_merged = df.merge(extra_df, on="name", how="left")
df_merged

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata,volcano_number,country,volcanic_landform,last_eruption_year
0,node,58959434,Chimborazo,NaN,-1.469082,-78.815950,6268,stratovolcano,dormant,Q14081,352071.0,Ecuador,Composite,550
1,node,58981085,Illiniza Sur,NaN,-0.663012,-78.715850,5248,stratovolcano,extinct,Q734280,NaN,NaN,NaN,NaN
2,node,59073041,Gunung Bromo,Mount Bromo,-7.942069,112.952977,2329,stratovolcano,active,Q679590,NaN,NaN,NaN,NaN
3,node,247716925,Mount Oku,NaN,6.199469,10.518025,3011,NaN,NaN,Q3322001,NaN,NaN,NaN,NaN
4,node,249407661,Volcán Alcedo,NaN,-0.396276,-91.109064,1130,shield,active,Q1424979,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1205,way,1545489827,Kawah Keramikan,Keramikan Crater,NaN,NaN,NaN,maar,active,NaN,NaN,NaN,NaN,NaN
1206,relation,5740787,Tukuran Volcanic Field,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1207,relation,9841413,Mount Bagana,NaN,NaN,NaN,NaN,NaN,NaN,Q799827,NaN,NaN,NaN,NaN
1208,relation,15002257,Bilate River Field,NaN,NaN,NaN,1700,NaN,extinct,Q860406,221291.0,Ethiopia,Cluster,None


In [22]:
df[df["name"].apply(lambda x: isinstance(x,str))]

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata
0,node,58959434,Chimborazo,NaN,-1.469082,-78.815950,6268,stratovolcano,dormant,Q14081
1,node,58981085,Illiniza Sur,NaN,-0.663012,-78.715850,5248,stratovolcano,extinct,Q734280
2,node,59073041,Gunung Bromo,Mount Bromo,-7.942069,112.952977,2329,stratovolcano,active,Q679590
3,node,247716925,Mount Oku,NaN,6.199469,10.518025,3011,NaN,NaN,Q3322001
4,node,249407661,Volcán Alcedo,NaN,-0.396276,-91.109064,1130,shield,active,Q1424979
...,...,...,...,...,...,...,...,...,...,...
2173,way,1545489827,Kawah Keramikan,Keramikan Crater,NaN,NaN,NaN,maar,active,NaN
2174,relation,5740787,Tukuran Volcanic Field,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2175,relation,9841413,Mount Bagana,NaN,NaN,NaN,NaN,NaN,NaN,Q799827
2176,relation,15002257,Bilate River Field,NaN,NaN,NaN,1700,NaN,extinct,Q860406


In [37]:
load_gvmid_stations("./Monitoring WorldWide System package from wovodat.zip")

NameError: name 'load_gvmid_stations' is not defined